In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

print("Project root:", ROOT)

INTERACTIONS = ROOT / "data" / "processed" / "mind" / "interactions.parquet"

interactions = pd.read_parquet(INTERACTIONS)

print(interactions.shape)
print(interactions["timestamp"].min())
print(interactions["timestamp"].max())

Project root: c:\Users\HP\OneDrive\Documents\IIIT\IRE\Assignment\News-Recommender
(5843444, 5)
2019-11-09 00:00:19
2019-11-14 23:59:13


In [2]:
daily_counts = (
    interactions
    .assign(date=interactions["timestamp"].dt.date)
    .groupby("date")
    .agg(
        interactions=("article_id", "size"),
        impressions=("impression_id", "nunique"),
        users=("user_id", "nunique"),
    )
)

print(daily_counts)

            interactions  impressions  users
date                                        
2019-11-09        524529        13570   9865
2019-11-10        520041        15048  10726
2019-11-11       1166308        32799  21298
2019-11-12       1178464        33654  22011
2019-11-13       1231673        31624  20884
2019-11-14       1222429        30270  20179


In [4]:
print("\nEarliest impression:")
print(interactions["timestamp"].min())

print("\nLatest impression:")
print(interactions["timestamp"].max())

print("\nUnique impressions:")
print(interactions["impression_id"].nunique())

print("\nUnique users:")
print(interactions["user_id"].nunique())


Earliest impression:
2019-11-09 00:00:19

Latest impression:
2019-11-14 23:59:13

Unique impressions:
156965

Unique users:
50000


In [5]:
print("\nImpressions per day:")
print(
    interactions
    .groupby(interactions["timestamp"].dt.date)["impression_id"]
    .nunique()
)


Impressions per day:
timestamp
2019-11-09    13570
2019-11-10    15048
2019-11-11    32799
2019-11-12    33654
2019-11-13    31624
2019-11-14    30270
Name: impression_id, dtype: int64


In [11]:
impression_times = (
    interactions[
        ["impression_id", "user_id", "timestamp"]
    ]
    .drop_duplicates("impression_id")
)

print(impression_times.shape)
print(impression_times.head())

(156965, 3)
   impression_id user_id           timestamp
0              1  U13740 2019-11-11 09:05:58
2              2  U91836 2019-11-12 18:11:30
13             3  U73700 2019-11-14 07:01:48
49             4  U34670 2019-11-11 05:28:05
53             5   U8125 2019-11-12 16:11:21


In [10]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

HISTORY_PATH = (
    ROOT
    / "data"
    / "processed"
    / "mind"
    / "user_history.parquet"
)

history = pd.read_parquet(HISTORY_PATH)

print("Shape:", history.shape)
print("\nColumns:")
print(history.columns.tolist())

print("\nData types:")
print(history.dtypes)

print("\nFirst 10 rows:")
display(history.head(10))

Shape: (5107639, 5)

Columns:
['user_id', 'timestamp', 'article_id', 'history_position', 'impression_id']

Data types:
user_id                        str
timestamp           datetime64[us]
article_id                     str
history_position             int64
impression_id                  str
dtype: object

First 10 rows:


,user_id,timestamp,article_id,history_position,impression_id
0,U13740,2019-11-11 09:05:58,N55189,0,1
1,U13740,2019-11-11 09:05:58,N42782,1,1
2,U13740,2019-11-11 09:05:58,N34694,2,1
3,U13740,2019-11-11 09:05:58,N45794,3,1
4,U13740,2019-11-11 09:05:58,N18445,4,1
5,U13740,2019-11-11 09:05:58,N63302,5,1
6,U13740,2019-11-11 09:05:58,N10414,6,1
7,U13740,2019-11-11 09:05:58,N19347,7,1
8,U13740,2019-11-11 09:05:58,N31801,8,1
9,U91836,2019-11-12 18:11:30,N31739,0,2


In [7]:
print("\nTimestamp range:")
print("Minimum:", history["timestamp"].min())
print("Maximum:", history["timestamp"].max())

print("\nUnique users:")
print(history["user_id"].nunique())

print("\nUnique articles:")
print(history["article_id"].nunique())

print("\nMissing values:")
print(history.isna().sum())


Timestamp range:
Minimum: 2019-11-09 00:00:19
Maximum: 2019-11-14 23:59:13

Unique users:
49108

Unique articles:
33195

Missing values:
user_id             0
timestamp           0
article_id          0
history_position    0
impression_id       0
dtype: int64


In [8]:
print(
    history.groupby("user_id")["timestamp"]
    .agg(["min", "max", "count"])
    .head(10)
)

                        min                 max  count
user_id                                               
U100    2019-11-12 07:34:12 2019-11-12 07:34:12     10
U1000   2019-11-13 23:16:18 2019-11-14 22:37:21      9
U10001  2019-11-11 05:30:21 2019-11-14 05:46:54     30
U10003  2019-11-11 07:06:02 2019-11-11 14:13:58     16
U10008  2019-11-09 14:13:33 2019-11-09 14:13:33     23
U10010  2019-11-13 09:53:30 2019-11-13 09:53:30      5
U10011  2019-11-10 08:31:14 2019-11-13 08:31:18     20
U10012  2019-11-09 15:55:58 2019-11-14 16:48:54     60
U10014  2019-11-13 07:04:55 2019-11-13 07:04:55      4
U10015  2019-11-13 09:47:37 2019-11-13 09:47:37      7


In [12]:
history_check = history.merge(
    impression_times[
        ["impression_id", "timestamp"]
    ],
    on="impression_id",
    how="left",
    suffixes=(
        "_history",
        "_impression"
    )
)

print(history_check.shape)

print(
    "\nMissing impression timestamps:",
    history_check["timestamp_impression"].isna().sum()
)

(5107639, 6)

Missing impression timestamps: 0


In [13]:
timestamp_mismatch = (
    history_check["timestamp_history"]
    != history_check["timestamp_impression"]
)

print(
    "Timestamp mismatches:",
    timestamp_mismatch.sum()
)

Timestamp mismatches: 0


In [14]:
user_check = history.merge(
    impression_times[
        ["impression_id", "user_id"]
    ],
    on="impression_id",
    how="left",
    suffixes=(
        "_history",
        "_impression"
    )
)

print(
    "User mismatches:",
    (
        user_check["user_id_history"]
        != user_check["user_id_impression"]
    ).sum()
)

User mismatches: 0
